# 2.2 — Data Preparation (UMAP / Sklearn Imputation Pipeline)

This notebook implements the full data preparation pipeline for the **Hip Replacement** dataset using a **no-listwise-deletion** strategy. All missing values are resolved through principled imputation rather than row removal, to preserve as many observations as possible.

## Pipeline Overview

| Step | Operation | Applied Before / After Split |
|---|---|---|
| 1 | Verify indicator columns | Before |
| 2 | Calculate derived scores (Pre-Op Q Score, Post-Op Q Score, EQ5D profiles & indices) | Before |
| 3 | Remove irrelevant columns (Post-Op questions, Predicted, CSVYear, EQ VAS) | Before |
| 4 | Missingness visualisation — UMAP + DBSCAN clustering | Before |
| 5 | Train / Test split (80 / 20, seed=42) | — |
| 6 | Classify columns by imputation strategy | **After** |
| 7 | Sklearn imputation pipeline (Mode, MICE, Constant) — fit on train only | **After** |
| 8 | Reconstruct Polars DataFrames | **After** |
| 9 | Create binary outcome variable `OHS_Success` | **After** |
| 10 | Final validation — zero null values | **After** |
| 11 | Save train & test datasets | **After** |

In [4]:
# Import Required Libraries
import polars as pl
import numpy as np
import pandas as pd
from pathlib import Path

from umap import UMAP
from sklearn.cluster import DBSCAN
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

In [5]:
pl.Config.set_tbl_cols(-1)       # display all columns
pl.Config.set_tbl_rows(-1)       # display all rows
pl.Config.set_float_precision(2) # display 2 decimal places

polars.config.Config

## Setup — Load Data & Explore

In [6]:
# Load the Knee Replacement dataset
df = pl.read_parquet('./data/interim/2.0-preprocessing.parquet')

# Keep an untouched copy for reference
df_original = df.clone()

print(f'Dataset loaded — {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

Dataset loaded — 139,236 rows x 49 columns


Provider Code,Procedure,Revision Flag,Year,Age Band,Gender,Pre-Op Q Assisted,Pre-Op Q Assisted By,Pre-Op Q Symptom Period,Pre-Op Q Previous Surgery,Pre-Op Q Living Arrangements,Pre-Op Q Disability,Heart Disease,High Bp,Stroke,Circulation,Lung Disease,Diabetes,Kidney Disease,Nervous System,Liver Disease,Cancer,Depression,Arthritis,Pre-Op Q Mobility,Pre-Op Q Self-Care,Pre-Op Q Activity,Pre-Op Q Discomfort,Pre-Op Q Anxiety,Pre-Op Q EQ5D Index Profile,Pre-Op Q EQ5D Index,Post-Op Q EQ5D Index Profile,Post-Op Q EQ5D Index,Pre-Op Q EQ VAS,Post-Op Q EQ VAS,Knee Replacement Pre-Op Q Pain,Knee Replacement Pre-Op Q Night Pain,Knee Replacement Pre-Op Q Washing,Knee Replacement Pre-Op Q Transport,Knee Replacement Pre-Op Q Walking,Knee Replacement Pre-Op Q Standing,Knee Replacement Pre-Op Q Limping,Knee Replacement Pre-Op Q Kneeling,Knee Replacement Pre-Op Q Work,Knee Replacement Pre-Op Q Confidence,Knee Replacement Pre-Op Q Shopping,Knee Replacement Pre-Op Q Stairs,Knee Replacement Pre-Op Q Score,Knee Replacement Post-Op Q Score
i32,i32,u8,i32,i32,f32,u8,i32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u32,f32,u32,f32,u16,u16,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,f32,f32
1,1,0,1,null,null,2,1,4,2,2,1,0,0,0,0,0,0,0,0,0,0,1,0,2,2,2,3,2,22232,-0.02,29223,null,63,50,0,0,2,3,2,1,0,0,1,2,2,2,15.00,29.00
1,1,0,1,null,null,2,1,1,2,2,2,0,0,0,0,0,0,0,0,0,0,1,0,2,2,3,2,2,22322,0.19,11222,0.69,30,58,3,2,4,3,4,3,1,1,1,3,2,3,30.00,39.00
1,1,0,1,null,null,2,1,2,2,1,2,0,0,0,0,0,0,0,0,0,0,0,1,2,1,3,3,1,21331,0.10,11111,1.00,65,80,0,1,4,2,2,1,2,3,1,3,2,3,24.00,45.00
1,1,0,1,null,null,2,1,2,2,1,1,1,0,0,0,0,0,0,0,0,0,0,1,2,1,2,2,1,21221,0.69,11121,0.80,90,90,1,3,3,2,4,2,2,2,2,0,3,3,27.00,43.00
1,1,0,1,null,null,1,1,3,2,1,1,0,1,0,0,0,0,0,0,0,0,0,1,2,1,2,2,1,21221,0.69,21221,0.69,95,null,1,4,3,3,3,3,1,2,3,3,2,4,32.00,36.00


## Initial Missing Values Overview

The table below shows every column that contains at least one null value **before any cleaning or imputation** is applied.
Unlike the Manual pipeline (2.1), this pipeline preserves all rows and resolves missing values through imputation.

In [7]:
# Show missing values summary before any cleaning
null_counts  = df.null_count()
missing_cols = [c for c in df.columns if null_counts[c][0] > 0]

print(f'Columns with missing values : {len(missing_cols)} / {df.shape[1]}')
print(f'Total missing value slots   : {sum(null_counts[c][0] for c in missing_cols):,}')

missing_summary = pl.DataFrame({
    'column': missing_cols,
    'count':  [null_counts[c][0] for c in missing_cols],
    'pct':    [round(null_counts[c][0] / df.shape[0] * 100, 2) for c in missing_cols],
}).sort('count', descending=True)

missing_summary

Columns with missing values : 30 / 49
Total missing value slots   : 110,317


column,count,pct
str,i64,f64
"""Pre-Op Q EQ VAS""",12986,9.33
"""Age Band""",9402,6.75
"""Gender""",9402,6.75
"""Pre-Op Q EQ5D Index""",7500,5.39
"""Post-Op Q EQ VAS""",6571,4.72
"""Post-Op Q EQ5D Index""",6274,4.51
"""Pre-Op Q Disability""",5907,4.24
"""Pre-Op Q Discomfort""",5540,3.98
"""Knee Replacement Pre-Op Q Scor…",5517,3.96


## Step 1 — Verify Indicator Columns

Binary comorbidity indicators (0 = absent, 1 = present) should have no nulls;
value `9` (missing) was already replaced by `0` in the preceding preprocessing notebook.

In [8]:
# Verify binary comorbidity indicator columns have no residual missing values
indicator_columns = [
    'Arthritis', 'Cancer', 'Circulation', 'Depression', 'Diabetes',
    'Heart Disease', 'High Bp', 'Kidney Disease', 'Liver Disease',
    'Lung Disease', 'Nervous System', 'Stroke'
]

existing_cols  = [c for c in indicator_columns if c in df.columns]
not_found_cols = [c for c in indicator_columns if c not in df.columns]

if not_found_cols:
    print(f'WARNING — Columns NOT found in dataframe: {not_found_cols}')

null_check = df.select(existing_cols).null_count()
print('Null counts for indicator columns:')
print(null_check)

Null counts for indicator columns:
shape: (1, 12)
┌────────┬────────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ Arthri ┆ Cancer ┆ Circul ┆ Depre ┆ Diabe ┆ Heart ┆ High  ┆ Kidne ┆ Liver ┆ Lung  ┆ Nervo ┆ Strok │
│ tis    ┆ ---    ┆ ation  ┆ ssion ┆ tes   ┆ Disea ┆ Bp    ┆ y Dis ┆ Disea ┆ Disea ┆ us    ┆ e     │
│ ---    ┆ u32    ┆ ---    ┆ ---   ┆ ---   ┆ se    ┆ ---   ┆ ease  ┆ se    ┆ se    ┆ Syste ┆ ---   │
│ u32    ┆        ┆ u32    ┆ u32   ┆ u32   ┆ ---   ┆ u32   ┆ ---   ┆ ---   ┆ ---   ┆ m     ┆ u32   │
│        ┆        ┆        ┆       ┆       ┆ u32   ┆       ┆ u32   ┆ u32   ┆ u32   ┆ ---   ┆       │
│        ┆        ┆        ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆ u32   ┆       │
╞════════╪════════╪════════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╡
│ 0      ┆ 0      ┆ 0      ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     │
└────────┴────────┴────────┴───────┴─────

## Step 2 — Calculate Derived Scores

Missing values for score and index columns are **derived deterministically** from other observed columns in the same row.
Applied before the train/test split — no leakage.

| Column | Derivation |
|---|---|
| `Knee Replacement Pre-Op Q Score` | Sum of 12 pre-op dimension columns |
| `Knee Replacement Post-Op Q Score` | Sum of 12 post-op dimension columns (null if any component is null) |
| `Pre-Op Q EQ5D Index Profile` | Concatenate 5 pre-op EQ-5D dimension columns |
| `Pre-Op Q EQ5D Index` | Calculated from profile using UK value set |
| `Post-Op Q EQ5D Index Profile` | Concatenate 5 post-op EQ-5D dimension columns |
| `Post-Op Q EQ5D Index` | Calculated from profile using UK value set |

In [9]:
# Calculate missing Pre-Op Q Score by summing the 12 pre-op dimension columns
t0_hr_cols = [
    c for c in df.columns
    if c.startswith('Knee Replacement Pre-Op Q') and c != 'Knee Replacement Pre-Op Q Score'
]
print(f'Pre-Op Q dimension columns ({len(t0_hr_cols)}): {t0_hr_cols}')

if len(t0_hr_cols) == 12:
    df = df.with_columns(
        pl.when(pl.col('Knee Replacement Pre-Op Q Score').is_null())
          .then(pl.sum_horizontal([pl.col(c) for c in t0_hr_cols]))
          .otherwise(pl.col('Knee Replacement Pre-Op Q Score'))
          .alias('Knee Replacement Pre-Op Q Score')
    )
    print(f"Remaining nulls in Pre-Op Q Score: {df['Knee Replacement Pre-Op Q Score'].null_count()}")
else:
    print(f'WARNING: Expected 12 columns, found {len(t0_hr_cols)}')

Pre-Op Q dimension columns (12): ['Knee Replacement Pre-Op Q Pain', 'Knee Replacement Pre-Op Q Night Pain', 'Knee Replacement Pre-Op Q Washing', 'Knee Replacement Pre-Op Q Transport', 'Knee Replacement Pre-Op Q Walking', 'Knee Replacement Pre-Op Q Standing', 'Knee Replacement Pre-Op Q Limping', 'Knee Replacement Pre-Op Q Kneeling', 'Knee Replacement Pre-Op Q Work', 'Knee Replacement Pre-Op Q Confidence', 'Knee Replacement Pre-Op Q Shopping', 'Knee Replacement Pre-Op Q Stairs']
Remaining nulls in Pre-Op Q Score: 0


In [10]:
# Calculate Post-Op Q Score by summing the 12 post-op dimension columns
# If any component is null -> score set to null (partial responses are invalid)
t1_hr_cols = [
    c for c in df.columns
    if c.startswith('Knee Replacement Post-Op Q')
    and c != 'Knee Replacement Post-Op Q Score'
    and 'Predicted' not in c
]
print(f'Post-Op Q dimension columns ({len(t1_hr_cols)}): {t1_hr_cols}')

null_check = df.select(t1_hr_cols).null_count()
print(f'Null counts in Post-Op Q dimension columns:\n{null_check}')

if len(t1_hr_cols) == 12:
    has_null = pl.any_horizontal([pl.col(c).is_null() for c in t1_hr_cols])
    df = df.with_columns(
        pl.when(has_null)
          .then(None)
          .when(pl.col('Knee Replacement Post-Op Q Score').is_null())
          .then(pl.sum_horizontal([pl.col(c) for c in t1_hr_cols]))
          .otherwise(pl.col('Knee Replacement Post-Op Q Score'))
          .alias('Knee Replacement Post-Op Q Score')
    )
    print(f"Remaining nulls in Post-Op Q Score: {df['Knee Replacement Post-Op Q Score'].null_count()}")
else:
    print(f'WARNING: Expected 12 columns, found {len(t1_hr_cols)}')

Post-Op Q dimension columns (0): []
Null counts in Post-Op Q dimension columns:
shape: (1, 0)
┌┐
╞╡
└┘


In [11]:
def calculate_eq5d_index(profile):
    """Calculate EQ-5D index from 5-digit profile string using the UK value set."""
    if profile is None or len(str(profile)) != 5:
        return None
    try:
        digits = [int(d) for d in str(profile)]
        if any(d not in [1, 2, 3] for d in digits):
            return None
    except (ValueError, TypeError):
        return None

    constant   = 0.081
    n3_penalty = 0.269
    dim_weights = {
        0: {'2': 0.069, '3': 0.314},  # Mobility
        1: {'2': 0.104, '3': 0.214},  # Self-care
        2: {'2': 0.036, '3': 0.094},  # Usual activities
        3: {'2': 0.123, '3': 0.386},  # Pain/discomfort
        4: {'2': 0.071, '3': 0.236},  # Anxiety/depression
    }

    total_deduction = constant
    has_level_3 = False
    for i, level in enumerate(digits):
        if level == 2:
            total_deduction += dim_weights[i]['2']
        elif level == 3:
            total_deduction += dim_weights[i]['3']
            has_level_3 = True

    if has_level_3:
        total_deduction += n3_penalty

    return round(1.0 - total_deduction, 6)

print('EQ-5D UK value set function defined.')

EQ-5D UK value set function defined.


In [12]:
# Fill missing Pre-Op Q EQ5D Index Profile by concatenating 5 pre-op EQ-5D components
eq5d_pre_cols = [
    'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity',
    'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety'
]

df = df.with_columns(
    pl.when(pl.col('Pre-Op Q EQ5D Index Profile').is_null())
      .then(pl.concat_str([pl.col(c).cast(pl.Utf8) for c in eq5d_pre_cols], separator=''))
      .otherwise(pl.col('Pre-Op Q EQ5D Index Profile'))
      .alias('Pre-Op Q EQ5D Index Profile')
)

df = df.with_columns(
    pl.when(pl.col('Pre-Op Q EQ5D Index').is_null())
      .then(
          pl.col('Pre-Op Q EQ5D Index Profile')
            .map_elements(calculate_eq5d_index, return_dtype=pl.Float64)
      )
      .otherwise(pl.col('Pre-Op Q EQ5D Index'))
      .alias('Pre-Op Q EQ5D Index')
)

print(f"Remaining nulls — Pre-Op Q EQ5D Index Profile : {df['Pre-Op Q EQ5D Index Profile'].null_count()}")
print(f"Remaining nulls — Pre-Op Q EQ5D Index         : {df['Pre-Op Q EQ5D Index'].null_count()}")

Remaining nulls — Pre-Op Q EQ5D Index Profile : 0
Remaining nulls — Pre-Op Q EQ5D Index         : 7500


In [14]:
#No need to run as these are future values
# Fill missing Post-Op Q EQ5D Index Profile by concatenating 5 post-op EQ-5D components
eq5d_post_cols = [
    'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity',
    'Post-Op Q Discomfort', 'Post-Op Q Anxiety'
]

df = df.with_columns(
    pl.when(pl.col('Post-Op Q EQ5D Index Profile').is_null())
      .then(pl.concat_str([pl.col(c).cast(pl.Utf8) for c in eq5d_post_cols], separator=''))
      .otherwise(pl.col('Post-Op Q EQ5D Index Profile'))
      .alias('Post-Op Q EQ5D Index Profile')
)

df = df.with_columns(
    pl.when(pl.col('Post-Op Q EQ5D Index').is_null())
      .then(
          pl.col('Post-Op Q EQ5D Index Profile')
            .map_elements(calculate_eq5d_index, return_dtype=pl.Float64)
      )
      .otherwise(pl.col('Post-Op Q EQ5D Index'))
      .alias('Post-Op Q EQ5D Index')
)

print(f"Remaining nulls — Post-Op Q EQ5D Index Profile : {df['Post-Op Q EQ5D Index Profile'].null_count()}")
print(f"Remaining nulls — Post-Op Q EQ5D Index         : {df['Post-Op Q EQ5D Index'].null_count()}")

ColumnNotFoundError: unable to find column "Post-Op Q Mobility"; valid columns: ["Provider Code", "Procedure", "Revision Flag", "Year", "Age Band", "Gender", "Pre-Op Q Assisted", "Pre-Op Q Assisted By", "Pre-Op Q Symptom Period", "Pre-Op Q Previous Surgery", "Pre-Op Q Living Arrangements", "Pre-Op Q Disability", "Heart Disease", "High Bp", "Stroke", "Circulation", "Lung Disease", "Diabetes", "Kidney Disease", "Nervous System", "Liver Disease", "Cancer", "Depression", "Arthritis", "Pre-Op Q Mobility", "Pre-Op Q Self-Care", "Pre-Op Q Activity", "Pre-Op Q Discomfort", "Pre-Op Q Anxiety", "Pre-Op Q EQ5D Index Profile", "Pre-Op Q EQ5D Index", "Post-Op Q EQ5D Index Profile", "Post-Op Q EQ5D Index", "Pre-Op Q EQ VAS", "Post-Op Q EQ VAS", "Knee Replacement Pre-Op Q Pain", "Knee Replacement Pre-Op Q Night Pain", "Knee Replacement Pre-Op Q Washing", "Knee Replacement Pre-Op Q Transport", "Knee Replacement Pre-Op Q Walking", "Knee Replacement Pre-Op Q Standing", "Knee Replacement Pre-Op Q Limping

## Step 3 — Remove Irrelevant Columns

Drop columns that are not available or appropriate at prediction time:

- **Post-Op Q individual questions**: not available pre-operatively.
- **Predicted columns**: regression-based predictions that must not be used as features.
- **`CSVYear`**: administrative bookkeeping, not clinically meaningful.
- **EQ VAS columns** (`Pre-Op Q EQ VAS`, `Post-Op Q EQ VAS`): self-reported VAS scores, not used in the model.

Columns **retained** from the post-op domain (outcome variables only):
- `Post-Op Q EQ5D Index Profile`
- `Post-Op Q EQ5D Index`
- `Knee Replacement Post-Op Q Score`

In [ ]:
# Identify columns to remove and keep
keep_post_op = [
    'Post-Op Q EQ5D Index Profile',
    'Post-Op Q EQ5D Index',
    'Knee Replacement Post-Op Q Score',
]

removed_columns = [
    col for col in df.columns
    if ('Post-Op' in col and col not in keep_post_op) or 'Predicted' in col or col == 'CSVYear'
]
kept_columns = [col for col in df.columns if col not in removed_columns]

print(f'Columns to remove ({len(removed_columns)}):')
print(removed_columns)
print(f'Columns to keep ({len(kept_columns)}):')
print(kept_columns)

Columns to remove (1):
['Post-Op Q EQ VAS']
Columns to keep (48):
['Provider Code', 'Procedure', 'Revision Flag', 'Year', 'Age Band', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis', 'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety', 'Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Post-Op Q EQ5D Index Profile', 'Post-Op Q EQ5D Index', 'Pre-Op Q EQ VAS', 'Knee Replacement Pre-Op Q Pain', 'Knee Replacement Pre-Op Q Night Pain', 'Knee Replacement Pre-Op Q Washing', 'Knee Replacement Pre-Op Q Transport', 'Knee Replacement Pre-Op Q Walking', 'Knee Replacement Pre-Op Q Standing', 'Knee Replacement Pre-Op Q Limping', 'Knee Replacement Pre-Op Q Knee

In [ ]:
# Apply column selection and drop EQ VAS
df = df.select(kept_columns)

vas_cols = [c for c in ['Pre-Op Q EQ VAS', 'Post-Op Q EQ VAS'] if c in df.columns]
if vas_cols:
    df = df.drop(vas_cols)
    print(f'Dropped EQ VAS columns: {vas_cols}')

print(f'Dataframe shape after column removal: {df.shape}')

# Remaining nulls after column removal
null_counts_post = df.null_count()
remaining_missing = [c for c in df.columns if null_counts_post[c][0] > 0]
print(f'Columns still containing nulls ({len(remaining_missing)}):')
for c in remaining_missing:
    print(f'  {c}: {null_counts_post[c][0]:,} nulls ({null_counts_post[c][0]/df.shape[0]*100:.2f}%)')

Dropped EQ VAS columns: ['Pre-Op Q EQ VAS']
Dataframe shape after column removal: (139236, 47)
Columns still containing nulls (27):
  Age Band: 9,402 nulls (6.75%)
  Gender: 9,402 nulls (6.75%)
  Pre-Op Q Assisted: 1,511 nulls (1.09%)
  Pre-Op Q Symptom Period: 1,209 nulls (0.87%)
  Pre-Op Q Previous Surgery: 1,042 nulls (0.75%)
  Pre-Op Q Living Arrangements: 2,087 nulls (1.50%)
  Pre-Op Q Disability: 5,907 nulls (4.24%)
  Pre-Op Q Mobility: 4,286 nulls (3.08%)
  Pre-Op Q Self-Care: 4,410 nulls (3.17%)
  Pre-Op Q Activity: 4,489 nulls (3.22%)
  Pre-Op Q Discomfort: 5,540 nulls (3.98%)
  Pre-Op Q Anxiety: 5,161 nulls (3.71%)
  Pre-Op Q EQ5D Index: 7,500 nulls (5.39%)
  Post-Op Q EQ5D Index: 6,274 nulls (4.51%)
  Knee Replacement Pre-Op Q Pain: 198 nulls (0.14%)
  Knee Replacement Pre-Op Q Night Pain: 1,304 nulls (0.94%)
  Knee Replacement Pre-Op Q Washing: 125 nulls (0.09%)
  Knee Replacement Pre-Op Q Transport: 1,350 nulls (0.97%)
  Knee Replacement Pre-Op Q Walking: 1,443 nulls (1.04

## Step 4 — Missingness Visualisation (UMAP + DBSCAN)

Before splitting and imputing, we visualise the **structure of missingness** across rows using UMAP dimensionality reduction
on a binary missingness indicator matrix. DBSCAN clusters reveal groups of patients with similar missing-data patterns,
informing imputation strategy choices.

In [ ]:
# Build binary missingness indicator matrix (1 = missing, 0 = present)
def is_missing_np(series: pl.Series) -> np.ndarray:
    if series.dtype in (pl.String, pl.Categorical):
        return (
            series.cast(pl.String).is_null() | series.cast(pl.String).is_in(['', '*'])
        ).to_numpy().astype(np.uint8)
    return series.is_null().to_numpy().astype(np.uint8)

col_names   = df.columns
miss_matrix = np.column_stack([is_missing_np(df[c]) for c in col_names])

print(f'Missingness matrix shape   : {miss_matrix.shape}')
print(f'Overall missingness rate   : {miss_matrix.mean()*100:.2f}%')
print(f'Rows with >= 1 missing val : {(miss_matrix.sum(axis=1) > 0).sum():,}')

Missingness matrix shape   : (139236, 47)
Overall missingness rate   : 1.30%
Rows with >= 1 missing val : 31,793


In [ ]:
# Fit UMAP on missingness matrix, then cluster with DBSCAN
# init='random' avoids the spectral initialisation failure caused by the near-zero
# variance in a sparse missingness matrix (most rows have no missing values).
reducer   = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1, init='random')
embedding = reducer.fit_transform(miss_matrix)

dbscan         = DBSCAN(eps=0.5, min_samples=10)
cluster_labels = dbscan.fit_predict(embedding)

cluster_ids         = np.unique(cluster_labels)
cluster_sizes       = {cid: int((cluster_labels == cid).sum()) for cid in cluster_ids}
cluster_avg_missing = {
    cid: float(miss_matrix[cluster_labels == cid].mean()) * 100 for cid in cluster_ids
}

print(f'DBSCAN clusters found: {len(cluster_ids)} (noise label = -1 if present)')
for cid in sorted(cluster_ids):
    label = 'Noise' if cid == -1 else f'Cluster {cid}'
    print(f'  {label}: {cluster_sizes[cid]:,} rows, avg missing: {cluster_avg_missing[cid]:.1f}%')

c:\Users\mittall\source\EAISI\NHS\EAISI-NHS\.venv\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


DBSCAN clusters found: 2132 (noise label = -1 if present)
  Cluster 0: 287 rows, avg missing: 6.5%
  Cluster 1: 7,930 rows, avg missing: 4.3%
  Cluster 2: 61 rows, avg missing: 26.2%
  Cluster 3: 263 rows, avg missing: 17.3%
  Cluster 4: 136 rows, avg missing: 6.5%
  Cluster 5: 340 rows, avg missing: 6.9%
  Cluster 6: 60 rows, avg missing: 6.5%
  Cluster 7: 46 rows, avg missing: 9.7%
  Cluster 8: 193 rows, avg missing: 9.5%
  Cluster 9: 144 rows, avg missing: 27.2%
  Cluster 10: 52 rows, avg missing: 6.4%
  Cluster 11: 134 rows, avg missing: 10.4%
  Cluster 12: 232 rows, avg missing: 22.0%
  Cluster 13: 48 rows, avg missing: 0.0%
  Cluster 14: 43 rows, avg missing: 0.0%
  Cluster 15: 54 rows, avg missing: 0.0%
  Cluster 16: 60 rows, avg missing: 0.0%
  Cluster 17: 4,277 rows, avg missing: 2.2%
  Cluster 18: 54 rows, avg missing: 0.0%
  Cluster 19: 62 rows, avg missing: 0.0%
  Cluster 20: 63 rows, avg missing: 0.0%
  Cluster 21: 43 rows, avg missing: 0.0%
  Cluster 22: 38 rows, avg miss

In [15]:
# Build hover text per cluster then render interactive Plotly scatter
cluster_top_missing = {}
for cid in cluster_ids:
    mask      = cluster_labels == cid
    miss_pcts = miss_matrix[mask].mean(axis=0) * 100
    top_cols  = sorted(zip(col_names, miss_pcts), key=lambda x: -x[1])[:10]
    top_str   = '<br>'.join([f'  {c}: {p:.1f}%' for c, p in top_cols if p > 0]) or 'none'
    cluster_top_missing[cid] = top_str

sorted_ids = sorted(cluster_ids.tolist(), key=lambda c: (1, 0) if c == -1 else (0, -cluster_sizes[c]))
palette    = px.colors.qualitative.Dark24

cluster_color_map, cluster_name_map = {}, {}
for i, cid in enumerate(sorted_ids):
    avg_miss, size = cluster_avg_missing[cid], cluster_sizes[cid]
    if avg_miss == 0:
        color, name = 'green', f'Cluster {cid} — no missing (n={size:,})'
    elif cid == -1:
        color, name = '#aaaaaa', f'Noise (n={size:,}, avg miss={avg_miss:.1f}%)'
    else:
        color, name = palette[i % len(palette)], f'Cluster {cid} (n={size:,}, avg miss={avg_miss:.1f}%)'
    cluster_color_map[cid] = color
    cluster_name_map[cid]  = name

plot_df = pd.DataFrame({
    'UMAP1':        embedding[:, 0],
    'UMAP2':        embedding[:, 1],
    'cluster_name': [cluster_name_map[c] for c in cluster_labels],
    'missing_pct':  (miss_matrix.sum(axis=1) / miss_matrix.shape[1] * 100).round(1),
    'top_missing':  [cluster_top_missing[c] for c in cluster_labels],
})
plot_df['cluster_name'] = pd.Categorical(
    plot_df['cluster_name'],
    categories=[cluster_name_map[cid] for cid in sorted_ids], ordered=True
)
color_map = {cluster_name_map[cid]: cluster_color_map[cid] for cid in sorted_ids}

fig = px.scatter(
    plot_df.sort_values('cluster_name'), x='UMAP1', y='UMAP2',
    color='cluster_name', color_discrete_map=color_map,
    category_orders={'cluster_name': [cluster_name_map[cid] for cid in sorted_ids]},
    custom_data=['cluster_name', 'missing_pct', 'top_missing'],
    labels={'cluster_name': 'Cluster (largest first)'},
    opacity=0.6,
)
fig.update_traces(
    marker=dict(size=4),
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>Row missing cols: %{customdata[1]}%<br>'
        '<br><b>Top missing columns:</b><br>%{customdata[2]}<extra></extra>'
    ),
)
for cid in sorted_ids:
    if cid == -1:
        continue
    mask   = cluster_labels == cid
    cx, cy = float(embedding[mask, 0].mean()), float(embedding[mask, 1].mean())
    fig.add_trace(go.Scatter(
        x=[cx], y=[cy], mode='markers',
        marker=dict(symbol='circle', size=10, color=cluster_color_map[cid],
                    line=dict(width=2, color='black')),
        hovertext=(
            f'<b>Cluster {cid} — Centroid</b><br>Size: {cluster_sizes[cid]:,}<br>'
            f'Avg missing: {cluster_avg_missing[cid]:.1f}%<br><br>'
            f'<b>Top missing columns:</b><br>{cluster_top_missing[cid]}'
        ),
        hoverinfo='text', showlegend=False, name=f'Cluster {cid} centroid',
    ))
fig.update_layout(
    title='UMAP of Missingness Matrix — DBSCAN Clusters (hover for details)',
    legend_title='Cluster (largest first)',
    legend=dict(itemclick='toggleothers', itemdoubleclick='toggle'),
    width=950, height=680,
)
display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))

NameError: name 'cluster_ids' is not defined

## Step 5 — Train / Test Split

The data is split **here**, before any imputation is applied. This prevents data leakage: imputation values
(modes, MICE fits) are learned **only** from the training set, then applied identically to the test set.

**Split:** 80% train / 20% test — seed 42.

In [ ]:
# Convert to pandas for sklearn, then shuffle and split 80 / 20
df_pd = df.to_pandas()

df_shuffled  = df_pd.sample(frac=1.0, random_state=42)
split_n      = int(len(df_shuffled) * 0.8)
df_train_raw = df_shuffled.iloc[:split_n].reset_index(drop=True)
df_test_raw  = df_shuffled.iloc[split_n:].reset_index(drop=True)

print(f'Full dataset : {len(df_pd):,} rows x {len(df_pd.columns)} columns')
print(f'Train set    : {len(df_train_raw):,} rows  ({100*split_n/len(df_shuffled):.1f}%)')
print(f'Test set     : {len(df_test_raw):,} rows  ({100*(len(df_shuffled)-split_n)/len(df_shuffled):.1f}%)')

Full dataset : 139,236 rows x 47 columns
Train set    : 111,388 rows  (80.0%)
Test set     : 27,848 rows  (20.0%)


In [ ]:
# Show null counts per split before imputation
train_missing = df_train_raw.isnull().sum()
test_missing  = df_test_raw.isnull().sum()

train_missing = train_missing[train_missing > 0]
test_missing  = test_missing[test_missing > 0]

print(f'Train — columns with nulls before imputation: {len(train_missing)}')
print(train_missing.to_string())
print(f'\nTest  — columns with nulls before imputation: {len(test_missing)}')
print(test_missing.to_string())

Train — columns with nulls before imputation: 27
Age Band                                7508
Gender                                  7508
Pre-Op Q Assisted                       1222
Pre-Op Q Symptom Period                  961
Pre-Op Q Previous Surgery                815
Pre-Op Q Living Arrangements            1637
Pre-Op Q Disability                     4749
Pre-Op Q Mobility                       3425
Pre-Op Q Self-Care                      3518
Pre-Op Q Activity                       3583
Pre-Op Q Discomfort                     4410
Pre-Op Q Anxiety                        4128
Pre-Op Q EQ5D Index                     5985
Post-Op Q EQ5D Index                    5013
Knee Replacement Pre-Op Q Pain           160
Knee Replacement Pre-Op Q Night Pain    1055
Knee Replacement Pre-Op Q Washing        108
Knee Replacement Pre-Op Q Transport     1091
Knee Replacement Pre-Op Q Walking       1163
Knee Replacement Pre-Op Q Standing      1168
Knee Replacement Pre-Op Q Limping       1139
Knee R

## Step 6 — Classify Columns by Imputation Strategy

Each column with missing values is assigned to one of the following imputation strategies:

| Strategy | Columns | Rationale |
|---|---|---|
| **Mode** (most frequent) | Binary comorbidity indicators, `Pre-Op Q Assisted`, `Pre-Op Q Previous Surgery` | Low missingness; dominant category expected |
| **Constant 9** | `Pre-Op Q Living Arrangements`, `Pre-Op Q Disability` | Domain knowledge: 9 = Unknown / Not Disclosed |
| **MICE** (Iterative Imputer) | All remaining numeric columns with nulls | Multivariate relationships preserve correlations |
| **Pass-through** | Numeric columns without any nulls | No imputation needed |

In [ ]:
_numeric_kinds = {
    'int8','int16','int32','int64',
    'uint8','uint16','uint32','uint64',
    'float32','float64'
}

# Binary comorbidity indicators -> MODE
indicator_cols = [c for c in [
    'Arthritis', 'Cancer', 'Circulation', 'Depression', 'Diabetes',
    'Heart Disease', 'High Bp', 'Kidney Disease', 'Liver Disease',
    'Lung Disease', 'Nervous System', 'Stroke'
] if c in df_pd.columns]

# Low-missingness ordinal columns -> MODE
mode_cols = [c for c in ['Pre-Op Q Assisted', 'Pre-Op Q Previous Surgery'] if c in df_pd.columns]

# Domain-knowledge constant fill (9 = Unknown / Not Disclosed)
const9_cols = [c for c in ['Pre-Op Q Living Arrangements', 'Pre-Op Q Disability'] if c in df_pd.columns]

_assigned = set(indicator_cols + mode_cols + const9_cols)

# Remaining numeric columns with any nulls -> MICE
mice_cols = [
    c for c in df_pd.columns
    if c not in _assigned and df_pd[c].dtype.name in _numeric_kinds and df_pd[c].isnull().any()
]

# Numeric columns without nulls -> pass-through
passthrough_cols = [
    c for c in df_pd.columns
    if c not in _assigned and c not in mice_cols and df_pd[c].dtype.name in _numeric_kinds
]

# Non-numeric columns excluded from pipeline, rejoined after imputation
non_numeric_cols = [c for c in df_pd.columns if df_pd[c].dtype.name not in _numeric_kinds]

# Ordered output of ColumnTransformer
all_numeric_cols = indicator_cols + mode_cols + mice_cols + const9_cols + passthrough_cols

print(f'Mode — binary indicators : {len(indicator_cols)} cols')
print(f'Mode — ordinal           : {mode_cols}')
print(f'Constant 9               : {const9_cols}')
print(f'MICE — numeric with nulls: {mice_cols}')
print(f'Pass-through (no nulls)  : {len(passthrough_cols)} cols')
print(f'Non-numeric (rejoined)   : {non_numeric_cols}')

Mode — binary indicators : 12 cols
Mode — ordinal           : ['Pre-Op Q Assisted', 'Pre-Op Q Previous Surgery']
Constant 9               : ['Pre-Op Q Living Arrangements', 'Pre-Op Q Disability']
MICE — numeric with nulls: ['Age Band', 'Gender', 'Pre-Op Q Symptom Period', 'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety', 'Pre-Op Q EQ5D Index', 'Post-Op Q EQ5D Index', 'Knee Replacement Pre-Op Q Pain', 'Knee Replacement Pre-Op Q Night Pain', 'Knee Replacement Pre-Op Q Washing', 'Knee Replacement Pre-Op Q Transport', 'Knee Replacement Pre-Op Q Walking', 'Knee Replacement Pre-Op Q Standing', 'Knee Replacement Pre-Op Q Limping', 'Knee Replacement Pre-Op Q Kneeling', 'Knee Replacement Pre-Op Q Work', 'Knee Replacement Pre-Op Q Confidence', 'Knee Replacement Pre-Op Q Shopping', 'Knee Replacement Pre-Op Q Stairs', 'Knee Replacement Post-Op Q Score']
Pass-through (no nulls)  : 7 cols
Non-numeric (rejoined)   : ['Pre-Op Q EQ5D Index Profil

## Step 7 — Sklearn Imputation Pipeline

A single `sklearn.Pipeline` applies all numeric imputation steps with **no data leakage**:
the pipeline is fitted on the training set only, then applied to both train and test sets.

Non-numeric columns (EQ5D profile strings) are excluded from the pipeline and rejoined after imputation.

In [ ]:
imputer_ct = ColumnTransformer(
    transformers=[
        ('mode',   SimpleImputer(strategy='most_frequent'),          indicator_cols + mode_cols),
        ('mice',   IterativeImputer(random_state=42, max_iter=10),   mice_cols),
        ('const9', SimpleImputer(strategy='constant', fill_value=9), const9_cols),
        ('pass',   'passthrough',                                     passthrough_cols),
    ],
    verbose_feature_names_out=False,
    remainder='drop',
)

pipeline = Pipeline([('imputer', imputer_ct)])
print('Imputation pipeline built.')

Imputation pipeline built.


In [ ]:
print('Fitting imputation pipeline on training data (MICE may take a moment)...')
X_train_imputed = pipeline.fit_transform(df_train_raw)

print('Transforming test data...')
X_test_imputed  = pipeline.transform(df_test_raw)

print(f'Imputed train shape  : {X_train_imputed.shape}')
print(f'Imputed test shape   : {X_test_imputed.shape}')
print(f'Remaining NaN — train: {int(np.isnan(X_train_imputed).sum())}')
print(f'Remaining NaN — test : {int(np.isnan(X_test_imputed).sum())}')

Fitting imputation pipeline on training data (MICE may take a moment)...


c:\Users\mittall\source\EAISI\NHS\EAISI-NHS\.venv\lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Transforming test data...
Imputed train shape  : (111388, 46)
Imputed test shape   : (27848, 46)
Remaining NaN — train: 0
Remaining NaN — test : 0


## Step 8 — Reconstruct Polars DataFrames

In [ ]:
# Rebuild pandas DataFrames with correct column names, rejoin non-numeric columns
train_imputed_pd = pd.DataFrame(X_train_imputed, columns=all_numeric_cols)
test_imputed_pd  = pd.DataFrame(X_test_imputed,  columns=all_numeric_cols)

for col in non_numeric_cols:
    train_imputed_pd[col] = df_train_raw[col].values
    test_imputed_pd[col]  = df_test_raw[col].values

# Convert back to Polars
df_train = pl.from_pandas(train_imputed_pd)
df_test  = pl.from_pandas(test_imputed_pd)

print(f'Train DataFrame : {df_train.shape[0]:,} rows x {df_train.shape[1]} columns')
print(f'Test  DataFrame : {df_test.shape[0]:,} rows x {df_test.shape[1]} columns')
print(f'Null count — Train: {sum(df_train.null_count().row(0))}')
print(f'Null count — Test : {sum(df_test.null_count().row(0))}')

Train DataFrame : 111,388 rows x 47 columns
Test  DataFrame : 27,848 rows x 47 columns
Null count — Train: 0
Null count — Test : 0


## Step 9 — Create Outcome Variable (`health_gain`)

Using the Oxford Hip Score (OHS) strategy, a patient is classified as a **success (1)** if **either** condition is met:

1. **Improvement >= 6 points**: Post-Op Score − Pre-Op Score >= 6 *(MCID — recent studies)*
2. **Absolute threshold reached**: Post-Op Score >= 26 *(minimum acceptable outcome)*

If neither condition is satisfied → **failure (0)**.

> In this pipeline, rows with originally missing Post-Op scores are **retained** and imputed via MICE before applying this rule.

In [ ]:

# Calculate health_gain = Post-Op Q Score − Pre-Op Q Score (outcome variable to predict)
health_gain_expr = (
    pl.col('Knee Replacement Post-Op Q Score') - pl.col('Knee Replacement Pre-Op Q Score')
).alias('health_gain')

df_train = df_train.with_columns(health_gain_expr)
df_test  = df_test.with_columns(health_gain_expr)

print(f"'health_gain' column added.")
print(f"  Train — mean: {df_train['health_gain'].mean():.2f}, min: {df_train['health_gain'].min()}, max: {df_train['health_gain'].max()}")
print(f"  Test  — mean: {df_test['health_gain'].mean():.2f},  min: {df_test['health_gain'].min()},  max: {df_test['health_gain'].max()}")

# Drop Knee Replacement Post-Op Q Score — no longer needed after health_gain is calculated.
# Retaining it would cause target leakage (it directly determines health_gain).
for col in ['Knee Replacement Post-Op Q Score']:
    if col in df_train.columns:
        df_train = df_train.drop(col)
        df_test  = df_test.drop(col)
        print(f"Dropped '{col}' from train and test sets (target leakage prevention).")

print(f"\nTrain shape: {df_train.shape}")
print(f"Test  shape: {df_test.shape}")


ColumnNotFoundError: unable to find column "Knee Replacement Post-Op Q Score"; valid columns: ["Arthritis", "Cancer", "Circulation", "Depression", "Diabetes", "Heart Disease", "High Bp", "Kidney Disease", "Liver Disease", "Lung Disease", "Nervous System", "Stroke", "Pre-Op Q Assisted", "Pre-Op Q Previous Surgery", "Age Band", "Gender", "Pre-Op Q Symptom Period", "Pre-Op Q Mobility", "Pre-Op Q Self-Care", "Pre-Op Q Activity", "Pre-Op Q Discomfort", "Pre-Op Q Anxiety", "Pre-Op Q EQ5D Index", "Post-Op Q EQ5D Index", "Knee Replacement Pre-Op Q Pain", "Knee Replacement Pre-Op Q Night Pain", "Knee Replacement Pre-Op Q Washing", "Knee Replacement Pre-Op Q Transport", "Knee Replacement Pre-Op Q Walking", "Knee Replacement Pre-Op Q Standing", "Knee Replacement Pre-Op Q Limping", "Knee Replacement Pre-Op Q Kneeling", "Knee Replacement Pre-Op Q Work", "Knee Replacement Pre-Op Q Confidence", "Knee Replacement Pre-Op Q Shopping", "Knee Replacement Pre-Op Q Stairs", "Pre-Op Q Living Arrangements", "

## Step 10 — Final Validation — No Missing Values

Verify that both train and test sets contain **zero null values** before saving.
The UMAP pipeline uses imputation throughout — no rows were dropped.

In [ ]:
# Verify no nulls remain in train set
train_nulls   = df_train.null_count()
train_missing = [c for c in df_train.columns if train_nulls[c][0] > 0]

print(f'Train — columns with nulls: {len(train_missing)}')
if train_missing:
    print(train_nulls.select(train_missing))
else:
    print('No missing values in training set. OK')
print(f'Train shape: {df_train.shape}')

Train — columns with nulls: 0
No missing values in training set. OK
Train shape: (111388, 47)


In [ ]:
# Verify no nulls remain in test set
test_nulls   = df_test.null_count()
test_missing = [c for c in df_test.columns if test_nulls[c][0] > 0]

print(f'Test  — columns with nulls: {len(test_missing)}')
if test_missing:
    print(test_nulls.select(test_missing))
else:
    print('No missing values in test set. OK')
print(f'Test shape: {df_test.shape}')

Test  — columns with nulls: 0
No missing values in test set. OK
Test shape: (27848, 47)


## Step 11 — Save Train & Test Datasets

In [ ]:
df_train.write_parquet('./data/interim/2.2-train.parquet', compression='gzip')
df_test.write_parquet('./data/interim/2.2-test.parquet', compression='gzip')

print(f'Training set saved -> ./data/interim/2.2-train.parquet — shape: {df_train.shape}')
print(f'Test set saved     -> ./data/interim/2.2-test.parquet  — shape: {df_test.shape}')

Training set saved -> ./data/interim/2.2-train.parquet — shape: (111388, 47)
Test set saved     -> ./data/interim/2.2-test.parquet  — shape: (27848, 47)
